# Frozen test benchmark: Seed1 and Seed2

Select a T4 GPU and Run all. This notebook evaluates eight trained checkpoints across the frozen 21 conditions and 19,962 official test images. It reports F1(frozen), F1@0.5 and mAP. It never trains or fits thresholds.

Seed0 detector outputs are reused only after checksums, binding, IDs, targets and geometry checks pass. Corrupted RGB inputs and transformed GT are reconstructed and checked against Seed0. Missing or changed inputs stop execution rather than silently rerunning detection or changing the protocol.

Both seeds share the same corruption seed and images. Prediction batches are saved atomically to Drive, and verified saved batches are skipped on reruns. A runtime reset requires Run all again. Do not run concurrent writers into the same output directory.

Checkpoint choice comes exclusively from saved clean-validation thresholds. Oracle is a separately trained GT-reference model, not a guaranteed upper bound. NME describes detector reliability; subgroup associations do not establish causality.


In [1]:
SEEDS = (1, 2)
MODEL_NAMES = ("r18", "r50", "m3", "oracle")
PROJECT_PATH = "/content/drive/MyDrive/FacialVisualProfile"
# Preserve the Seed0 inference batch size and prediction-shard boundaries.
BATCH_SIZE = 16
SHARD_SIZE = 256

## 1. Runtime and helpers

In [2]:
from google.colab import drive
from pathlib import Path
from collections import Counter
import base64
import csv
import gc
import hashlib
import importlib
import importlib.metadata
import importlib.util
import inspect
import io
import json
import math
import os
import platform
import shutil
import sys
import time
import traceback
import uuid
import zipfile

import numpy as np
from PIL import Image
import torch
from torch.torch_version import TorchVersion

assert SEEDS == (1, 2) and MODEL_NAMES == ("r18", "r50", "m3", "oracle")
assert BATCH_SIZE == 16 and SHARD_SIZE == 256
drive.mount("/content/drive")
PROJECT = Path(PROJECT_PATH)
WORKSPACE = Path("/content/multiseed_frozen_test_workspace")
DATA_ROOT = Path("/content/multiseed_frozen_test_data/raw")
OUTPUT_ROOT = PROJECT / "runs/benchmark/multiseed_seed1_seed2"
assert PROJECT.is_dir()
assert torch.cuda.is_available(), "Select the T4 GPU runtime before running"
torch.set_num_threads(2)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ENVIRONMENT = {name: importlib.metadata.version(name) for name in ("torch", "torchvision", "numpy", "Pillow", "scikit-learn")}
RUNTIME = {"python": platform.python_version(), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(), "gpu": torch.cuda.get_device_name(0)}
SOURCE_RUNS = ("r18_seed0", "r50_seed0", "m3_seed0")
RUNS = tuple(f"{model}_seed{seed}" for seed in SEEDS for model in MODEL_NAMES)

def encoded(value):
    return (json.dumps(value, sort_keys=True, indent=2, allow_nan=False) + "\n").encode("utf-8")

def sha(data):
    return hashlib.sha256(data).hexdigest()

def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_once(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_bytes() == data, f"Existing file differs; preserved: {path}"
        return
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    temporary.write_bytes(data)
    assert not path.exists(), f"Concurrent writer detected: {path}"
    temporary.replace(path)

def copy_verified(source, destination, expected):
    if Path(destination).exists():
        assert sha_file(destination) == expected
    else:
        assert sha_file(source) == expected
        write_once(destination, Path(source).read_bytes())
    assert sha_file(destination) == expected

print("Environment:", ENVIRONMENT)
print("Runtime:", RUNTIME)
print("No facenet-pytorch installation is needed: detector outputs are reused")

Mounted at /content/drive
Environment: {'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'numpy': '2.1.3', 'Pillow': '11.3.0', 'scikit-learn': '1.6.1'}
Runtime: {'python': '3.13.15', 'cuda': '13.0', 'cudnn': 92700, 'gpu': 'Tesla T4'}
No facenet-pytorch installation is needed: detector outputs are reused


## 2. Restore the original frozen protocol (embedded verified snapshot)

In [3]:
# Original Day19 handoff bytes; no manual upload is required.
FROZEN_ZIP_SHA256 = "b358598a17a9a1aa86c3f09f96778ab6c11a0b27bf1358562050838321c71eb5"
FROZEN_ZIP_BASE64 = "..."
frozen_bytes = base64.b64decode(FROZEN_ZIP_BASE64)
assert sha(frozen_bytes) == FROZEN_ZIP_SHA256
with zipfile.ZipFile(io.BytesIO(frozen_bytes)) as archive:
    assert archive.testzip() is None
    for name in archive.namelist():
        relative = Path(name)
        assert not relative.is_absolute() and ".." not in relative.parts
        write_once(WORKSPACE / relative, archive.read(name))
del frozen_bytes
CONFIG = json.loads((WORKSPACE / "configs/benchmark_v1.json").read_bytes())
FREEZE_PATH = WORKSPACE / "docs/results/benchmark_freeze/day17/freeze_audit.json"
FREEZE = json.loads(FREEZE_PATH.read_bytes())
assert FREEZE["passed"] is True
for relative, expected in FREEZE["source_and_artifact_sha256"].items():
    assert sha_file(WORKSPACE / relative) == expected, f"Frozen source or artifact differs: {relative}"
MANIFEST_PATH = WORKSPACE / CONFIG["official_test_manifest"]
assert sha_file(MANIFEST_PATH) == CONFIG["sample_manifest_sha256"]
with MANIFEST_PATH.open(newline="") as stream:
    TEST_NAMES = [row["sample_id"] for row in csv.DictReader(stream)]
assert len(TEST_NAMES) == len(set(TEST_NAMES)) == 19962
CONDITIONS = CONFIG["conditions"]
assert [(c["condition"], c["severity"]) for c in CONDITIONS] == ([('clean', 0)] + [(kind, severity) for kind in ('blur', 'brightness', 'rotation', 'occlusion', 'jpeg') for severity in range(1, 5)])
assert len(CONFIG["attribute_names"]) == 24
print("Day17 frozen rules, source hashes, test order and 21 conditions: PASS")

Day17 frozen rules, source hashes, test order and 21 conditions: PASS


## 3. Verify Seed0 reports and restore identical test JPEG bytes

In [4]:
REPORT_PINS = {'day19': {'day19_benchmark_report.json': '79e36c3725d10c2319206023f2b859854108fc7f1fc176e6998f0d96331b3e1b', 'execution_binding.json': '6e3088cdb07e6e6bc3edc966e19189a0c8ffed7c80ab9469c56a3dc78b6df065', 'test_jpeg_sha256.json': '3948ce5877a5d7a9b83513eeb562f53d6c86aa98e0975b46156e2f8559c120a4'}, 'day20': {'day20_benchmark_report.json': '33dbef0b2585be9a83fbd281121042c199a45efd4ef514bdbc5fae9e5a6c696a', 'execution_binding.json': '6c2f8d7724715baa354b499697d3e948a7dc1fefbc2f214aa2ce40faaeea8cca', 'test_jpeg_sha256.json': '3948ce5877a5d7a9b83513eeb562f53d6c86aa98e0975b46156e2f8559c120a4'}}
SOURCE_INFO = {}
for day in ("day19", "day20"):
    root = PROJECT / "runs/benchmark" / day
    for filename, expected in REPORT_PINS[day].items():
        assert (root / filename).is_file(), f"Missing Seed0 file: {root / filename}"
        assert sha_file(root / filename) == expected, f"Seed0 archived file differs: {filename}"
    report = json.loads((root / f"{day}_benchmark_report.json").read_bytes())
    binding = json.loads((root / "execution_binding.json").read_bytes())
    assert sha_file(root / "execution_binding.json") == report["execution_binding_sha256"]
    assert report["test_evaluated"] is True and report["threshold_fitting_performed"] is False
    for name, version in ENVIRONMENT.items():
        assert binding["environment"][name] == version, f"Seed0 inference environment differs: {name}"
    assert binding["python"] == RUNTIME["python"]
    assert binding["cuda_runtime"] == RUNTIME["cuda"]
    assert binding["cudnn_version"] == RUNTIME["cudnn"]
    assert binding["gpu"] == RUNTIME["gpu"]
    assert binding["batch_size"] == BATCH_SIZE and binding["shard_size"] == SHARD_SIZE
    assert binding["config_sha256"] == sha_file(WORKSPACE / "configs/benchmark_v1.json")
    assert binding["freeze_audit_sha256"] == sha_file(FREEZE_PATH)
    assert binding["manifest_sha256"] == sha_file(MANIFEST_PATH)
    assert binding["attribute_names"] == CONFIG["attribute_names"]
    SOURCE_INFO[day] = {"root": root, "report": report, "binding": binding}
assert SOURCE_INFO["day19"]["binding"]["annotation_sha256"] == SOURCE_INFO["day20"]["binding"]["annotation_sha256"]
assert SOURCE_INFO["day19"]["binding"]["image_zip_sha256"] == SOURCE_INFO["day20"]["binding"]["image_zip_sha256"]
assert SOURCE_INFO["day19"]["binding"]["detector_weights_sha256"] == SOURCE_INFO["day20"]["binding"]["detector_weights_sha256"]
assert SOURCE_INFO["day19"]["binding"]["conditions"] + SOURCE_INFO["day20"]["binding"]["conditions"] == CONDITIONS
JPEG_HASHES = json.loads((SOURCE_INFO["day19"]["root"] / "test_jpeg_sha256.json").read_bytes())
assert JPEG_HASHES == json.loads((SOURCE_INFO["day20"]["root"] / "test_jpeg_sha256.json").read_bytes())
assert set(JPEG_HASHES) == set(TEST_NAMES)
annotation_hashes = SOURCE_INFO["day19"]["binding"]["annotation_sha256"]
for filename, expected in annotation_hashes.items():
    copy_verified(PROJECT / "data" / filename, DATA_ROOT / filename, expected)
rows = [line.split() for line in (DATA_ROOT / "list_eval_partition.txt").read_text(encoding="utf-8-sig").splitlines()]
assert [name for name, split in rows if split == "2"] == TEST_NAMES
assert sha_file(DATA_ROOT / "list_eval_partition.txt") == FREEZE["partition_sha256"]
zip_hash = SOURCE_INFO["day19"]["binding"]["image_zip_sha256"]
candidates = [Path("/content/multiseed_shared_data/img_align_celeba.zip"), Path("/content/seed1_shared_data/img_align_celeba.zip"), DATA_ROOT.parent / "img_align_celeba.zip"]
image_archive = next((p for p in candidates if p.is_file()), candidates[-1])
if not image_archive.exists():
    print("Copying original image archive...", flush=True)
    image_archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = image_archive.with_name(image_archive.name + ".partial")
    shutil.copyfile(PROJECT / "data/img_align_celeba.zip", temporary)
    assert sha_file(temporary) == zip_hash
    temporary.replace(image_archive)
assert sha_file(image_archive) == zip_hash
image_root = DATA_ROOT / "img_align_celeba"
image_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(image_archive) as archive:
    members = {}
    for member in archive.infolist():
        if member.is_dir() or not member.filename.lower().endswith(".jpg"):
            continue
        name = Path(member.filename).name
        assert name not in members
        members[name] = member
    for index, name in enumerate(TEST_NAMES, 1):
        content = archive.read(members[name])
        assert sha(content) == JPEG_HASHES[name]
        write_once(image_root / name, content)
        if index % 2000 == 0 or index == len(TEST_NAMES):
            print(f"Verified test JPEGs: {index}/{len(TEST_NAMES)}", flush=True)
# Use the exact dataset snapshot used for Seed0 formal evaluation.
dataset_source = PROJECT / "runs/m3_seed0/source/datasets/celeba.py"
copy_verified(dataset_source, WORKSPACE / "datasets/celeba.py", SOURCE_INFO["day19"]["binding"]["dataset_source_sha256"])
for package in ("datasets", "evaluation"):
    (WORKSPACE / package / "__init__.py").touch(exist_ok=True)
for name in list(sys.modules):
    if any(name == p or name.startswith(p + ".") for p in ("datasets", "evaluation")):
        del sys.modules[name]
sys.path.insert(0, str(WORKSPACE))
os.chdir(WORKSPACE)
importlib.invalidate_caches()
from datasets.celeba import CelebAAttributes
from datasets import benchmark_preprocessing as preprocessing
from datasets import corruptions
from evaluation.benchmark_attributes import evaluate_benchmark_attributes as evaluate
DATASET = CelebAAttributes(split="test", root=DATA_ROOT)
assert DATASET.names == TEST_NAMES
assert DATASET.attribute_names == CONFIG["attribute_names"]
TARGETS = torch.stack([DATASET.targets[name] for name in TEST_NAMES]).numpy()
print("Official test IDs, labels, attributes and original image hashes: PASS")
print("No model inference performed")

Copying original image archive...
Verified test JPEGs: 2000/19962
Verified test JPEGs: 4000/19962
Verified test JPEGs: 6000/19962
Verified test JPEGs: 8000/19962
Verified test JPEGs: 10000/19962
Verified test JPEGs: 12000/19962
Verified test JPEGs: 14000/19962
Verified test JPEGs: 16000/19962
Verified test JPEGs: 18000/19962
Verified test JPEGs: 19962/19962
Official test IDs, labels, attributes and original image hashes: PASS
No model inference performed


## 4. Verify and reuse the Seed0 prediction shards

In [5]:
def load_source_shard(path, binding, condition, names, targets):
    with zipfile.ZipFile(path) as archive:
        assert len(archive.namelist()) == 3
        assert set(archive.namelist()) == {"manifest.json", "records.json", "predictions.npz"}
        manifest = json.loads(archive.read("manifest.json"))
        assert manifest["binding_sha256"] == binding
        assert manifest["condition"] == condition
        assert set(manifest["files"]) == {"records.json", "predictions.npz"}
        payload = {key: archive.read(key) for key in manifest["files"]}
        for key, data in payload.items():
            assert sha(data) == manifest["files"][key], f"Shard checksum failed: {path}"
    with np.load(io.BytesIO(payload["predictions.npz"]), allow_pickle=False) as stored:
        arrays = {key: stored[key] for key in stored.files}
    records = json.loads(payload["records.json"])
    assert arrays["filenames"].tolist() == names
    np.testing.assert_array_equal(arrays["targets"], targets)
    assert len(records) == len(names)
    assert arrays["geometry_valid"].shape == (len(names),)
    assert arrays["geometry_valid"].dtype == np.bool_
    assert arrays["predicted_normalized_points"].shape == (len(names), 10)
    assert np.isfinite(arrays["predicted_normalized_points"]).all()
    for run in SOURCE_RUNS:
        logits = arrays[run + "_logits"]
        assert logits.shape == (len(names), 24) and logits.dtype == np.float32
        assert np.isfinite(logits).all()
    for index, (name, record) in enumerate(zip(names, records)):
        assert record["sample_id"] == name
        assert record["condition"] == condition["condition"]
        assert record["severity"] == condition["severity"]
        assert record["detection"]["status"] in ("ok", "no_face", "invalid_output")
        valid = record["detection"]["status"] == "ok"
        assert bool(arrays["geometry_valid"][index]) == valid
        if valid:
            assert record["nme"] is not None and np.isfinite(record["nme"])
        else:
            assert record["nme"] is None
            assert not arrays["predicted_normalized_points"][index].any()
    return arrays, records

SOURCE_INDEX = {}
verified = 0
for condition in CONDITIONS:
    label = f"{condition['condition']}_s{condition['severity']}"
    day = "day19" if condition["condition"] in ("clean", "blur", "brightness") else "day20"
    info = SOURCE_INFO[day]
    for start in range(0, len(TEST_NAMES), SHARD_SIZE):
        stop = min(start + SHARD_SIZE, len(TEST_NAMES))
        relative = f"shards/{label}/{start:06d}_{stop:06d}.zip"
        path = info["root"] / relative
        assert path.is_file(), f"Missing Seed0 detector shard: {path}"
        expected = info["report"]["shard_sha256"][relative]
        assert sha_file(path) == expected, f"Seed0 shard differs: {path}"
        arrays, records = load_source_shard(path, info["report"]["execution_binding_sha256"], condition, TEST_NAMES[start:stop], TARGETS[start:stop])
        SOURCE_INDEX[(label, start)] = {"path": path, "sha256": expected, "day": day, "binding_sha256": info["report"]["execution_binding_sha256"]}
        del arrays, records
        verified += 1
        if verified % 100 == 0:
            print("Verified Seed0 shards:", verified, flush=True)
assert verified == 1638
print("Verified Seed0 shards: 1638/1638")
print("Detector coordinates and masks will be reused; no detector rerun")

Verified Seed0 shards: 100
Verified Seed0 shards: 200
Verified Seed0 shards: 300
Verified Seed0 shards: 400
Verified Seed0 shards: 500
Verified Seed0 shards: 600
Verified Seed0 shards: 700
Verified Seed0 shards: 800
Verified Seed0 shards: 900
Verified Seed0 shards: 1000
Verified Seed0 shards: 1100
Verified Seed0 shards: 1200
Verified Seed0 shards: 1300
Verified Seed0 shards: 1400
Verified Seed0 shards: 1500
Verified Seed0 shards: 1600
Verified Seed0 shards: 1638/1638
Detector coordinates and masks will be reused; no detector rerun


## 5. Strictly load the eight selected checkpoints and frozen thresholds

In [6]:
MODELS, THRESHOLDS, MODEL_PROVENANCE = {}, {}, {}
for seed in SEEDS:
    for kind in MODEL_NAMES:
        run = f"{kind}_seed{seed}"
        root = PROJECT / "runs" / run
        threshold_path = PROJECT / "runs/benchmark_preparation" / f"seed{seed}" / f"{run}.json"
        threshold = json.loads(threshold_path.read_bytes())
        training = json.loads((root / "config.json").read_bytes())
        audit = json.loads((root / "validation_export_audit.json").read_bytes())
        assert training["run_name"] == threshold["run_name"] == run
        assert training["seed"] == seed
        assert threshold["fit_split"] == "valid" and threshold["samples"] == 19867
        assert threshold["test_evaluated"] is False and audit["passed"] is True
        assert threshold["grid"] == [i / 100 for i in range(5, 96)]
        assert threshold["tie_break"] == "closest_to_0.5_then_lower" and threshold["comparison"] == ">="
        assert threshold["attribute_names"] == CONFIG["attribute_names"]
        assert len(threshold["thresholds"]) == 24 and all(t in threshold["grid"] for t in threshold["thresholds"])
        checkpoint_path = root / threshold["checkpoint"]
        print(run, "verifying checkpoint SHA256...", flush=True)
        for path, key in ((root / "config.json", "config_sha256"), (checkpoint_path, "checkpoint_sha256"), (root / "validation_predictions.npz", "prediction_sha256")):
            assert sha_file(path) == threshold[key] == audit[key], f"Training binding differs: {run}/{key}"
        for relative, expected in training["source_sha256"].items():
            assert sha_file(root / "source" / relative) == expected
        source_relative = {"r18": "models/resnet18.py", "r50": "models/resnet50.py", "m3": "models/resnet18_geometry.py", "oracle": "models/resnet18_geometry.py"}[kind]
        source = root / "source" / source_relative
        assert training["source_sha256"][source_relative] == SOURCE_INFO["day19"]["binding"]["models"][f"{kind if kind != 'oracle' else 'm3'}_seed0"]["model_source_sha256"]
        spec = importlib.util.spec_from_file_location(f"frozen_{run}", source)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        cls = getattr(module, {"r18": "ResNet18Attributes", "r50": "ResNet50Attributes", "m3": "ResNet18GeometryAttributes", "oracle": "ResNet18GeometryAttributes"}[kind])
        model = cls(num_attributes=24, pretrained=False).to("cuda")
        state_key = "model_state" if kind in ("r18", "r50") else "model_state_dict"
        with torch.serialization.safe_globals([TorchVersion]):
            state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        assert state["config"] == training and state["epoch"] == threshold["selected_epoch"]
        model.load_state_dict(state[state_key], strict=True)
        model.eval()
        assert all(p.dtype == torch.float32 and p.device.type == "cuda" for p in model.parameters())
        MODELS[run] = model
        THRESHOLDS[run] = threshold
        MODEL_PROVENANCE[run] = {"checkpoint": threshold["checkpoint"], "selected_epoch": threshold["selected_epoch"], "checkpoint_sha256": threshold["checkpoint_sha256"], "config_sha256": threshold["config_sha256"], "threshold_sha256": sha_file(threshold_path), "prediction_sha256": threshold["prediction_sha256"], "source_sha256": training["source_sha256"], "state_key": state_key}
        del state
        print(run, "strict loading and frozen thresholds: PASS", flush=True)
print("Eight selected checkpoints loaded; no threshold fitting performed")

r18_seed1 verifying checkpoint SHA256...
r18_seed1 strict loading and frozen thresholds: PASS
r50_seed1 verifying checkpoint SHA256...
r50_seed1 strict loading and frozen thresholds: PASS
m3_seed1 verifying checkpoint SHA256...
m3_seed1 strict loading and frozen thresholds: PASS
oracle_seed1 verifying checkpoint SHA256...
oracle_seed1 strict loading and frozen thresholds: PASS
r18_seed2 verifying checkpoint SHA256...
r18_seed2 strict loading and frozen thresholds: PASS
r50_seed2 verifying checkpoint SHA256...
r50_seed2 strict loading and frozen thresholds: PASS
m3_seed2 verifying checkpoint SHA256...
m3_seed2 strict loading and frozen thresholds: PASS
oracle_seed2 verifying checkpoint SHA256...
oracle_seed2 strict loading and frozen thresholds: PASS
Eight selected checkpoints loaded; no threshold fitting performed


## 6. Resumable inference and paired-input checks

In [7]:
def assert_parameters_equal(actual, expected, path="parameters"):
    if isinstance(actual, dict):
        assert isinstance(expected, dict) and set(actual) == set(expected), path
        for key in actual:
            assert_parameters_equal(actual[key], expected[key], path + "." + key)
    elif isinstance(actual, (list, tuple)):
        assert isinstance(expected, (list, tuple)) and len(actual) == len(expected), path
        for index, (a, b) in enumerate(zip(actual, expected)):
            assert_parameters_equal(a, b, path + f"[{index}]")
    elif isinstance(actual, float) or isinstance(expected, float):
        assert math.isclose(float(actual), float(expected), rel_tol=0, abs_tol=1e-10), path
    else:
        assert actual == expected, path

def regenerate_inputs(condition, names, source_arrays, records):
    images, oracle_points = [], []
    maximum_gt_difference = 0.0
    for index, (name, record) in enumerate(zip(names, records)):
        raw = (DATA_ROOT / "img_align_celeba" / name).read_bytes()
        assert sha(raw) == JPEG_HASHES[name], f"Test JPEG changed: {name}"
        with Image.open(io.BytesIO(raw)) as image:
            canvas, gt = preprocessing.prepare_canvas(image, DATASET.landmarks[name])
        image, gt, metadata = corruptions.apply_corruption(canvas, condition["implementation_kind"], condition["severity"], sample_id=name, seed=CONFIG["corruption_seed"], gt_pixels=gt)
        assert corruptions.rgb_sha256(image) == record["final_rgb_sha256"], f"Final RGB differs: {name}"
        assert_parameters_equal(metadata["parameters"], record["corruption_parameters"])
        gt = np.asarray(gt, dtype=np.float64)
        delta = float(np.max(np.abs(gt - np.asarray(record["gt_pixels"], dtype=np.float64))))
        assert delta <= 1e-9, f"Transformed GT differs: {name}"
        maximum_gt_difference = max(maximum_gt_difference, delta)
        valid = bool(source_arrays["geometry_valid"][index])
        if valid:
            points = preprocessing.normalize_points(record["detection"]["landmarks"]).numpy()
            np.testing.assert_allclose(points, source_arrays["predicted_normalized_points"][index], rtol=0, atol=1e-6)
            eye_distance = float(np.linalg.norm(gt[0] - gt[1]))
            assert eye_distance > 0
            nme = float(np.linalg.norm(np.asarray(record["detection"]["landmarks"], dtype=np.float64) - gt, axis=1).mean() / eye_distance)
            assert abs(nme - record["nme"]) <= 1e-9
        else:
            assert not source_arrays["predicted_normalized_points"][index].any()
            assert record["nme"] is None
        images.append(preprocessing.normalize_canvas(image))
        oracle_points.append(preprocessing.normalize_points(gt))
    images = torch.stack(images)
    oracle_points = torch.stack(oracle_points)
    assert torch.isfinite(images).all() and torch.isfinite(oracle_points).all()
    return images, oracle_points, maximum_gt_difference

def save_predictions(path, arrays, records, binding_sha, source_sha, condition, gt_difference):
    buffer = io.BytesIO()
    np.savez_compressed(buffer, **arrays)
    payload = {"predictions.npz": buffer.getvalue(), "records.json": encoded(records)}
    manifest = {"binding_sha256": binding_sha, "source_shard_sha256": source_sha, "condition": condition, "maximum_gt_difference_px": gt_difference, "files": {name: sha(data) for name, data in payload.items()}}
    result = io.BytesIO()
    with zipfile.ZipFile(result, "w", zipfile.ZIP_DEFLATED) as archive:
        for name, data in payload.items():
            archive.writestr(name, data)
        archive.writestr("manifest.json", encoded(manifest))
    write_once(path, result.getvalue())

def load_predictions(path, names, targets, binding_sha, source_sha, condition):
    with zipfile.ZipFile(path) as archive:
        assert len(archive.namelist()) == 3 and set(archive.namelist()) == {"predictions.npz", "records.json", "manifest.json"}
        manifest = json.loads(archive.read("manifest.json"))
        assert manifest["binding_sha256"] == binding_sha and manifest["source_shard_sha256"] == source_sha
        assert manifest["condition"] == condition
        assert set(manifest["files"]) == {"predictions.npz", "records.json"}
        payload = {name: archive.read(name) for name in manifest["files"]}
        for name, data in payload.items():
            assert sha(data) == manifest["files"][name]
    with np.load(io.BytesIO(payload["predictions.npz"]), allow_pickle=False) as stored:
        arrays = {key: stored[key].copy() for key in stored.files}
    assert arrays["filenames"].tolist() == names
    np.testing.assert_array_equal(arrays["targets"], targets)
    records = json.loads(payload["records.json"])
    assert len(records) == len(names)
    assert arrays["geometry_valid"].shape == (len(names),) and arrays["geometry_valid"].dtype == np.bool_
    for key in ("predicted_normalized_points", "gt_normalized_points"):
        assert arrays[key].shape == (len(names), 10) and np.isfinite(arrays[key]).all()
    for index, record in enumerate(records):
        assert record["sample_id"] == names[index]
        assert record["condition"] == condition["condition"] and record["severity"] == condition["severity"]
        valid = record["detection"]["status"] == "ok"
        assert bool(arrays["geometry_valid"][index]) == valid
        if not valid:
            assert not arrays["predicted_normalized_points"][index].any()
    for run in RUNS:
        values = arrays[run + "_logits"]
        assert values.shape == (len(names), 24) and values.dtype == np.float32 and np.isfinite(values).all()
    assert manifest["maximum_gt_difference_px"] <= 1e-9
    return arrays, records, manifest

def run_benchmark(binding_sha):
    assert not torch.is_autocast_enabled(), "FP32 inference requires autocast disabled"
    started = time.perf_counter()
    summaries, shard_hashes = [], {}
    active = {}
    processed = 0
    try:
        for condition in CONDITIONS:
            label = f"{condition['condition']}_s{condition['severity']}"
            print("\n===", label, "===", flush=True)
            parts = {run: [] for run in RUNS}
            status_counts = Counter()
            nme_values = []
            maximum_gt_difference = 0.0
            for start in range(0, len(TEST_NAMES), SHARD_SIZE):
                stop = min(start + SHARD_SIZE, len(TEST_NAMES))
                names = TEST_NAMES[start:stop]
                source = SOURCE_INDEX[(label, start)]
                path = OUTPUT_ROOT / "shards" / label / f"{start:06d}_{stop:06d}.zip"
                active = {"condition": label, "start": start, "stop": stop}
                if not path.exists():
                    assert sha_file(source["path"]) == source["sha256"], "Seed0 source shard changed"
                    old, records = load_source_shard(source["path"], source["binding_sha256"], condition, names, TARGETS[start:stop])
                    images, gt_points, gt_difference = regenerate_inputs(condition, names, old, records)
                    predicted = torch.from_numpy(old["predicted_normalized_points"])
                    valid = torch.from_numpy(old["geometry_valid"])
                    arrays = {"filenames": np.asarray(names), "targets": TARGETS[start:stop],
                              "geometry_valid": old["geometry_valid"], "predicted_normalized_points": old["predicted_normalized_points"],
                              "gt_normalized_points": gt_points.numpy()}
                    for run in RUNS:
                        kind = run.split("_seed")[0]
                        batches = []
                        with torch.inference_mode():
                            for offset in range(0, len(names), BATCH_SIZE):
                                end = min(offset + BATCH_SIZE, len(names))
                                image_batch = images[offset:end].to("cuda")
                                if kind in ("r18", "r50"):
                                    logits = MODELS[run](image_batch)
                                else:
                                    point_batch = (predicted if kind == "m3" else gt_points)[offset:end].to("cuda")
                                    mask = valid[offset:end].to("cuda") if kind == "m3" else torch.ones(end-offset, dtype=torch.bool, device="cuda")
                                    logits = MODELS[run](image_batch, point_batch, mask)
                                    if kind == "m3" and (~mask).any():
                                        for replacement in (1e6, float("nan")):
                                            altered = point_batch.clone()
                                            altered[~mask] = replacement
                                            check = MODELS[run](image_batch, altered, mask)
                                            assert torch.equal(check, logits), f"Invalid-coordinate invariance failed: {run}"
                                assert logits.shape == (end-offset, 24) and torch.isfinite(logits).all()
                                batches.append(logits.cpu().numpy())
                        arrays[run + "_logits"] = np.concatenate(batches)
                    save_predictions(path, arrays, records, binding_sha, source["sha256"], condition, gt_difference)
                    del old, images, gt_points, predicted, valid, arrays, records, batches
                arrays, records, manifest = load_predictions(path, names, TARGETS[start:stop], binding_sha, source["sha256"], condition)
                shard_hashes[path.relative_to(OUTPUT_ROOT).as_posix()] = sha_file(path)
                maximum_gt_difference = max(maximum_gt_difference, manifest["maximum_gt_difference_px"])
                for record in records:
                    status_counts[record["detection"]["status"]] += 1
                    if record["nme"] is not None:
                        nme_values.append(record["nme"])
                for run in RUNS:
                    parts[run].append(arrays[run + "_logits"])
                processed += len(names)
                if stop % 2048 == 0 or stop == len(TEST_NAMES):
                    elapsed = (time.perf_counter() - started) / 60
                    print(f"{label}: {stop}/{len(TEST_NAMES)} verified/saved; elapsed={elapsed:.1f} min", flush=True)
                del arrays, records
            summary = {"condition": condition, "sample_count": len(TEST_NAMES), "detector_status_counts": dict(status_counts),
                       "detector_nme": {"valid_samples": len(nme_values), "mean": float(np.mean(nme_values)) if nme_values else None, "unit": "fraction of GT inter-eye distance"},
                       "maximum_gt_difference_px": maximum_gt_difference, "models": {}}
            assert sum(status_counts.values()) == 19962
            source_day = "day19" if condition["condition"] in ("clean", "blur", "brightness") else "day20"
            reference_summary = next(item for item in SOURCE_INFO[source_day]["report"]["conditions"] if item["condition"] == condition)
            for status in ("ok", "no_face", "invalid_output"):
                assert status_counts[status] == reference_summary["detector"][status]
            if nme_values:
                assert abs(float(np.mean(nme_values)) - reference_summary["detector"]["mean_nme"]) <= 1e-10
            for run in RUNS:
                logits = np.concatenate(parts[run])
                frozen = evaluate(logits, TARGETS, THRESHOLDS[run]["thresholds"])
                fixed = evaluate(logits, TARGETS, [0.5] * 24)
                assert abs(frozen["map"] - fixed["map"]) <= 1e-12
                summary["models"][run] = {"frozen_thresholds": frozen, "fixed_0_5": fixed}
                print(f"{run}: F1(frozen)={frozen['macro_f1']:.6f}; F1@0.5={fixed['macro_f1']:.6f}; mAP={frozen['map']:.6f}", flush=True)
            summaries.append(summary)
            write_once(OUTPUT_ROOT / "condition_reports" / f"{label}.json", encoded(summary))
            del parts, logits
        assert len(summaries) == 21 and len(shard_hashes) == 1638
        for summary in summaries:
            for run in RUNS:
                for rule in ("frozen_thresholds", "fixed_0_5"):
                    clean = summaries[0]["models"][run][rule]
                    current = summary["models"][run][rule]
                    summary["models"][run][rule + "_degradation_pp"] = {name: 100 * (clean[name] - current[name]) for name in ("macro_f1", "map")}
        report = {"seeds": list(SEEDS), "runs": list(RUNS), "attribute_names": CONFIG["attribute_names"],
                  "execution_binding_sha256": binding_sha, "test_sample_count": 19962, "condition_count": 21,
                  "unique_inputs": 419202, "model_sample_evaluations": 3353616,
                  "conditions": summaries, "shard_sha256": shard_hashes,
                  "detector_outputs_reused_from_seed0": True, "threshold_fitting_performed": False,
                  "test_evaluated": True, "full_benchmark_complete": True}
        write_once(OUTPUT_ROOT / "multiseed_benchmark_report.json", encoded(report))
        print("Seed1 and Seed2 four-model 21-condition test benchmark: COMPLETE")
        return report
    except Exception:
        error = {"binding_sha256": binding_sha, "active": active, "traceback": traceback.format_exc(), "completed_batches_preserved": True}
        write_once(OUTPUT_ROOT / "runtime_errors" / (uuid.uuid4().hex + ".json"), encoded(error))
        print("Execution stopped; verified completed shards remain on Drive", flush=True)
        raise

## 7. Bind this execution before test inference

In [8]:
FUNCTION_NAMES = ("load_source_shard", "assert_parameters_equal", "regenerate_inputs", "save_predictions", "load_predictions", "run_benchmark")
runner_source = "\n\n".join(inspect.getsource(globals()[name]) for name in FUNCTION_NAMES).encode("utf-8")
binding = {"seeds": list(SEEDS), "runs": list(RUNS), "models": MODEL_PROVENANCE,
           "runner_sha256": sha(runner_source), "frozen_zip_sha256": FROZEN_ZIP_SHA256,
           "base_config_sha256": sha_file(WORKSPACE / "configs/benchmark_v1.json"),
           "base_freeze_sha256": sha_file(FREEZE_PATH), "test_manifest_sha256": sha_file(MANIFEST_PATH),
           "source_reports": REPORT_PINS, "environment": ENVIRONMENT, "runtime": RUNTIME,
           "conditions": CONDITIONS, "corruption_seed": CONFIG["corruption_seed"],
           "batch_size": BATCH_SIZE, "shard_size": SHARD_SIZE,
           "precision": "FP32", "tf32": False, "cudnn_deterministic": True, "cudnn_benchmark": False,
           "detector_outputs": "Verified Seed0 prediction shards; no detector rerun",
           "failure_policy": "retain every sample; mask invalid M3 geometry; no GT substitution for M3",
           "threshold_fitting_performed": False}
write_once(OUTPUT_ROOT / "execution_binding.json", encoded(binding))
write_once(OUTPUT_ROOT / "source/inference_functions.py", runner_source)
write_once(OUTPUT_ROOT / "source/day19_frozen_inputs.zip", base64.b64decode(FROZEN_ZIP_BASE64))
write_once(OUTPUT_ROOT / "test_jpeg_sha256.json", encoded(JPEG_HASHES))
BINDING_SHA256 = sha(encoded(binding))
# Smoke checks use the first test batch only, after the protocol is bound.
condition = CONDITIONS[0]
source = SOURCE_INDEX[("clean_s0", 0)]
old, records = load_source_shard(source["path"], source["binding_sha256"], condition, TEST_NAMES[:SHARD_SIZE], TARGETS[:SHARD_SIZE])
# Image reproduction is checked before inference begins.
images, gt_points, _ = regenerate_inputs(condition, TEST_NAMES[:BATCH_SIZE], {"geometry_valid": old["geometry_valid"][:BATCH_SIZE], "predicted_normalized_points": old["predicted_normalized_points"][:BATCH_SIZE]}, records[:BATCH_SIZE])
with torch.inference_mode():
    for run in RUNS:
        kind = run.split("_seed")[0]
        rgb = images.to("cuda")
        if kind in ("r18", "r50"):
            values = MODELS[run](rgb)
        else:
            points = torch.from_numpy(old["predicted_normalized_points"][:BATCH_SIZE]).to("cuda") if kind == "m3" else gt_points.to("cuda")
            mask = torch.from_numpy(old["geometry_valid"][:BATCH_SIZE]).to("cuda") if kind == "m3" else torch.ones(BATCH_SIZE, dtype=torch.bool, device="cuda")
            values = MODELS[run](rgb, points, mask)
        assert values.shape == (BATCH_SIZE, 24) and torch.isfinite(values).all()
        print(run, "test forward: PASS")
del old, records, images, gt_points
print("Execution rules bound; no threshold fitting performed")

r18_seed1 test forward: PASS
r50_seed1 test forward: PASS
m3_seed1 test forward: PASS
oracle_seed1 test forward: PASS
r18_seed2 test forward: PASS
r50_seed2 test forward: PASS
m3_seed2 test forward: PASS
oracle_seed2 test forward: PASS
Execution rules bound; no threshold fitting performed


## 8. Run or resume all 21 conditions for both seeds

In [9]:
FINAL_REPORT = run_benchmark(BINDING_SHA256)


=== clean_s0 ===
clean_s0: 2048/19962 verified/saved; elapsed=0.2 min
clean_s0: 4096/19962 verified/saved; elapsed=0.5 min
clean_s0: 6144/19962 verified/saved; elapsed=0.7 min
clean_s0: 8192/19962 verified/saved; elapsed=1.0 min
clean_s0: 10240/19962 verified/saved; elapsed=1.2 min
clean_s0: 12288/19962 verified/saved; elapsed=1.4 min
clean_s0: 14336/19962 verified/saved; elapsed=1.9 min
clean_s0: 16384/19962 verified/saved; elapsed=2.1 min
clean_s0: 18432/19962 verified/saved; elapsed=2.3 min
clean_s0: 19962/19962 verified/saved; elapsed=2.4 min
r18_seed1: F1(frozen)=0.756272; F1@0.5=0.733832; mAP=0.821219
r50_seed1: F1(frozen)=0.756112; F1@0.5=0.730333; mAP=0.819387
m3_seed1: F1(frozen)=0.756069; F1@0.5=0.721822; mAP=0.819476
oracle_seed1: F1(frozen)=0.756939; F1@0.5=0.724789; mAP=0.820210
r18_seed2: F1(frozen)=0.756856; F1@0.5=0.730231; mAP=0.820792
r50_seed2: F1(frozen)=0.756335; F1@0.5=0.725858; mAP=0.819008
m3_seed2: F1(frozen)=0.757532; F1@0.5=0.731843; mAP=0.819305
oracle_seed

## 9. Export overall/per-attribute tables, compare seeds and archive reports

In [10]:
ANALYSIS = OUTPUT_ROOT / "analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)
def csv_bytes(rows):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue().encode("utf-8")
overall_rows, attribute_rows = [], []
for item in FINAL_REPORT["conditions"]:
    condition = item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    for run, metrics in item["models"].items():
        frozen, fixed = metrics["frozen_thresholds"], metrics["fixed_0_5"]
        overall_rows.append({"condition": label, "run_name": run, "seed": int(run.split("_seed")[1]), "model": run.split("_seed")[0],
                             "samples": 19962, "f1_frozen": frozen["macro_f1"], "f1_at_0.5": fixed["macro_f1"], "map": frozen["map"]})
        for index, attribute in enumerate(CONFIG["attribute_names"]):
            attribute_rows.append({"condition": label, "run_name": run, "attribute": attribute, "positive_count": frozen["positive_counts"][index],
                                   "threshold": THRESHOLDS[run]["thresholds"][index], "f1_frozen": frozen["per_attribute_f1"][index],
                                   "f1_at_0.5": fixed["per_attribute_f1"][index], "ap": frozen["per_attribute_ap"][index]})
assert len(overall_rows) == 168 and len(attribute_rows) == 4032
write_once(ANALYSIS / "overall_metrics.csv", csv_bytes(overall_rows))
write_once(ANALYSIS / "per_attribute_metrics.csv", csv_bytes(attribute_rows))
# Add the completed Seed0 reports to the same comparison table.
for day in ("day19", "day20"):
    for item in SOURCE_INFO[day]["report"]["conditions"]:
        condition = item["condition"]
        label = f"{condition['condition']}_s{condition['severity']}" if isinstance(condition, dict) else condition
        for run, metrics in item["models"].items():
            frozen, fixed = metrics["frozen_thresholds"], metrics["fixed_0_5"]
            overall_rows.append({"condition": label, "run_name": run, "seed": 0, "model": run.split("_seed")[0], "samples": 19962, "f1_frozen": frozen["macro_f1"], "f1_at_0.5": fixed["macro_f1"], "map": frozen["map"]})
# Oracle Seed0 is loaded separately because it was evaluated on Day21.
oracle0_path = PROJECT / "runs/benchmark/day21/day21_oracle_report.json"
if oracle0_path.is_file():
    oracle0 = json.loads(oracle0_path.read_bytes())
    write_once(OUTPUT_ROOT / "source/seed0_oracle_report.json", oracle0_path.read_bytes())
    print("Oracle Seed0 report preserved for the subsequent three-seed audit")
write_once(ANALYSIS / "seed0_rgb_m3_plus_seed1_seed2_metrics.csv", csv_bytes(overall_rows))
# Full three-seed Oracle and NME subgroup analysis are a separate audited step.
summary = {"seed1_seed2_complete": True, "overall_rows": 168, "attribute_rows": 4032,
           "all_prediction_shards_verified": 1638, "threshold_fitting_performed": False,
           "three_seed_oracle_comparison_and_nme_subgroups": "pending subsequent analysis"}
write_once(ANALYSIS / "completion_summary.json", encoded(summary))
payloads = {}
for folder in ("condition_reports", "analysis", "source"):
    for path in sorted((OUTPUT_ROOT / folder).rglob("*")):
        if path.is_file():
            payloads[path.relative_to(OUTPUT_ROOT).as_posix()] = path.read_bytes()
for name in ("execution_binding.json", "test_jpeg_sha256.json", "multiseed_benchmark_report.json"):
    payloads[name] = (OUTPUT_ROOT / name).read_bytes()
manifest = {"files": {name: sha(data) for name, data in payloads.items()}, "prediction_shards_included": False,
            "images_included": False, "checkpoints_included": False, "test_evaluated": True}
payloads["archive_manifest.json"] = encoded(manifest)
result = io.BytesIO()
with zipfile.ZipFile(result, "w", zipfile.ZIP_DEFLATED) as archive:
    for name, data in sorted(payloads.items()):
        info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, data)
archive_path = OUTPUT_ROOT / "seed1_seed2_test_results_archive.zip"
write_once(archive_path, result.getvalue())
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
print("Overall metric rows:", 168)
print("Per-attribute metric rows:", 4032)
print("Seed1 and Seed2 formal test archive: PASS")
print("Archive:", archive_path)
print("Archive SHA256:", sha_file(archive_path))
print("Download this executed notebook and the result archive for Git archiving")
from google.colab import files
files.download(str(archive_path))

Oracle Seed0 report preserved for the subsequent three-seed audit
Overall metric rows: 168
Per-attribute metric rows: 4032
Seed1 and Seed2 formal test archive: PASS
Archive: /content/drive/MyDrive/FacialVisualProfile/runs/benchmark/multiseed_seed1_seed2/seed1_seed2_test_results_archive.zip
Archive SHA256: 4c0cece5d688bbe87bb490c7f4c2429059a6ab40dae9c84b5cfd19ecb4baf45c
Download this executed notebook and the result archive for Git archiving


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Git handoff

Keep checkpoints, CelebA images and prediction shards on Drive. Download this executed notebook with outputs and `seed1_seed2_test_results_archive.zip`. Commit the notebook, report JSON/CSV files and source code in the local Git project using explicit paths. Do not add `runs/` or the image archive to Git.

This notebook completes formal inference for Seed1 and Seed2. A subsequent analysis can combine the already saved Seed0 Oracle report, audit all three seeds together, compute model differences and across-seed variability, and reconstruct NME subgroups from the shared records without further model inference.
